# v2 — Does Multi-Task Training Help the Adapter?

## Experiment Card

### 1. Hypothesis
If the adapter's AP improvement over the per-pair probe is caused by **multi-task
generalization** (shared representation across pairs) rather than just **non-linearity**
(MLP decision boundary vs linear classifier), then:

- A **single-task MLP** (same architecture, trained on one pair at a time) will
  score *between* the logistic regression and the multi-task adapter on data-scarce pairs.
- The multi-task adapter will outperform the single-task MLP by **> 5 AP** on
  data-scarce pairs (< 80 training samples), because only the multi-task model
  can draw on signal from the other 5 pairs.

**Falsified if:** the single-task MLP matches or exceeds the multi-task adapter on
data-scarce pairs — that would mean non-linearity, not multi-task transfer, is the cause.

### 2. Methods compared (one variable at a time)
| Method | Architecture | Training objective |
|---|---|---|
| Logistic regression (LR) | Linear | Single-pair |
| Single-task MLP (ST-MLP) | Non-linear (same as adapter) | Single-pair |
| Multi-task adapter (MT-MLP) | Non-linear (same architecture) | All 6 pairs jointly |

LR → ST-MLP isolates **non-linearity**.  
ST-MLP → MT-MLP isolates **multi-task transfer**.

### 3. Dependent variable
Per-pair AP and delta on the held-out 20% test set.

### 4. Controls
**All three methods use the same embeddings, the same train/test split (seed=42),
the same test images, and the same StandardScaler on the embeddings.**
- LR: `LogisticRegression(C=1.0, max_iter=2000)` on scaled embeddings
- ST-MLP: 1152→512→256→1, 30 epochs, AdamW lr=1e-3, same as adapter architecture
- MT-MLP: loaded from `adapter.pt`, trained by `train_adapter.py` with LR sweep
- Broken_intact: excluded from mean AP (n_pos=10, data-quality issue)

### 5. Baseline
- Chance: AP = 0.5
- Zero-shot SigLIP 2 on the same test crops

### 6. Success criteria
- ST-MLP delta over LR > +5 AP on scarce pairs → non-linearity matters
- MT-MLP delta over ST-MLP > +5 AP on scarce pairs → multi-task transfer confirmed
- MT-MLP delta over ST-MLP ≤ 0 on scarce pairs → multi-task hypothesis falsified

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.model_selection import train_test_split
from transformers import AutoProcessor, AutoModel

device = ('cuda' if torch.cuda.is_available()
          else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {device}')

DATA = Path('../../data')
PAIR_NAMES = ['full_empty','open_closed','on_off','cooked_raw','dirty_clean','broken_intact']
NUM_PAIRS  = len(PAIR_NAMES)
EMBED_DIM  = 1152
SMALL_N    = 80  # data-scarce threshold (pre-stated)

with open(DATA / 'sim_crops_all.pkl', 'rb') as f:
    samples = pickle.load(f)

images    = [s['image']    for s in samples]
labels    = np.array([s['label']    for s in samples])
pair_idxs = np.array([s['pair_idx'] for s in samples])

# Shared split — IDENTICAL for both methods
strat_key = pair_idxs * 2 + labels
idx = np.arange(len(samples))
idx_train, idx_test = train_test_split(idx, test_size=0.2, random_state=42, stratify=strat_key)

embeddings = np.load(DATA / 'embeddings_all_cache.npz')['embeddings']
X_train, y_train, p_train = embeddings[idx_train], labels[idx_train], pair_idxs[idx_train]
X_test,  y_test,  p_test  = embeddings[idx_test],  labels[idx_test],  pair_idxs[idx_test]
print(f'Train: {len(idx_train)}  Test: {len(idx_test)}')
for i, name in enumerate(PAIR_NAMES):
    m = pair_idxs == i
    n_tr = int((p_train == i).sum())
    scarce = 'SCARCE' if n_tr < SMALL_N else ''
    print(f'  {name:<16} n_train={n_tr:>3} {scarce}')

In [ ]:
# Zero-shot baseline on test crops
print('Loading SigLIP 2 for zero-shot...')
proc    = AutoProcessor.from_pretrained('google/siglip2-so400m-patch16-384')
zs_mdl  = AutoModel.from_pretrained('google/siglip2-so400m-patch16-384').to(device).eval()

PAIR_PROMPTS = {
    'full_empty':    ('a full object',   'an empty object'),
    'open_closed':   ('an open object',  'a closed object'),
    'on_off':        ('object turned on','object turned off'),
    'cooked_raw':    ('cooked food',     'raw food'),
    'dirty_clean':   ('a dirty object',  'a clean object'),
    'broken_intact': ('a broken object', 'an intact object'),
}

zs_aps = {}
for i, name in enumerate(PAIR_NAMES):
    mask = p_test == i
    if mask.sum() < 2 or len(np.unique(y_test[mask])) < 2: continue
    pos_t, neg_t = PAIR_PROMPTS[name]
    test_imgs = [images[idx_test[j]] for j in range(len(idx_test)) if p_test[j] == i]
    scores = []
    for b in range(0, len(test_imgs), 16):
        inp = proc(images=test_imgs[b:b+16], text=[pos_t, neg_t],
                   return_tensors='pt', padding='max_length', truncation=True).to(device)
        with torch.no_grad():
            logits = zs_mdl(**inp).logits_per_image
        scores.extend(torch.softmax(logits, dim=1)[:, 0].cpu().numpy())
    zs_aps[name] = average_precision_score(y_test[mask], scores)
    print(f'  {name:<16} zero-shot AP = {zs_aps[name]:.3f}')

del zs_mdl
if device == 'mps': torch.mps.empty_cache()

In [ ]:
# Method A: logistic regression per pair (linear, single-task)
# StandardScaler normalizes embedding dimensions — MLP handles varying magnitudes
# natively via learned weights; LR does not. Required for a fair comparison.
from sklearn.preprocessing import StandardScaler

scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

probe_aps = {}
for i, name in enumerate(PAIR_NAMES):
    tr_mask = p_train == i
    te_mask = p_test  == i
    if tr_mask.sum() < 4 or te_mask.sum() < 2: continue
    if len(np.unique(y_train[tr_mask])) < 2 or len(np.unique(y_test[te_mask])) < 2: continue
    clf = LogisticRegression(C=1.0, max_iter=2000).fit(X_train_s[tr_mask], y_train[tr_mask])
    probe_aps[name] = average_precision_score(
        y_test[te_mask], clf.predict_proba(X_test_s[te_mask])[:, 1])
    print(f'  {name:<16} LR probe AP = {probe_aps[name]:.3f}  (n_train={int(tr_mask.sum())})')

In [ ]:
# Method B: single-task MLP per pair (non-linear, single-task)
# Same architecture as the adapter (1152→512→256→1) but trained on one pair at a time.
# Isolates the non-linearity effect from multi-task transfer.
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import torch.nn.functional as F

ST_EPOCHS = 30
ST_LR     = 1e-3

class SingleTaskMLP(nn.Module):
    def __init__(self, in_dim=EMBED_DIM, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 512), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(512, 256),   nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, 1),
        )
    def forward(self, x): return self.net(x).squeeze(-1)

class PairDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).float()
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

stlm_aps = {}
for i, name in enumerate(PAIR_NAMES):
    tr_mask = p_train == i
    te_mask = p_test  == i
    if tr_mask.sum() < 4 or te_mask.sum() < 2: continue
    if len(np.unique(y_train[tr_mask])) < 2 or len(np.unique(y_test[te_mask])) < 2: continue

    m   = SingleTaskMLP().to(device)
    opt = AdamW(m.parameters(), lr=ST_LR, weight_decay=1e-3)
    sch = CosineAnnealingLR(opt, T_max=ST_EPOCHS)
    dl  = DataLoader(PairDataset(X_train[tr_mask], y_train[tr_mask]),
                     batch_size=32, shuffle=True)
    m.train()
    for _ in range(ST_EPOCHS):
        for xb, yb in dl:
            opt.zero_grad()
            F.binary_cross_entropy_with_logits(m(xb.to(device)), yb.to(device)).backward()
            opt.step()
        sch.step()

    m.eval()
    with torch.no_grad():
        scores = torch.sigmoid(m(torch.from_numpy(X_test[te_mask]).float().to(device))).cpu().numpy()
    stlm_aps[name] = average_precision_score(y_test[te_mask], scores)
    print(f'  {name:<16} ST-MLP AP   = {stlm_aps[name]:.3f}  (n_train={int(tr_mask.sum())})')

In [ ]:
# Method B: MLP adapter loaded from adapter.pt
class StateAdapter(nn.Module):
    def __init__(self, in_dim=EMBED_DIM, n=NUM_PAIRS, d=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 512), nn.GELU(), nn.Dropout(d),
            nn.Linear(512, 256),   nn.GELU(), nn.Dropout(d),
            nn.Linear(256, n),
        )
    def forward(self, x): return self.net(x)

adapter = StateAdapter().to(device)
adapter.load_state_dict(torch.load(DATA / 'adapter.pt', map_location=device))
adapter.eval()

class StateDataset(Dataset):
    def __init__(self, X, y, p):
        self.X=torch.from_numpy(X).float(); self.y=torch.from_numpy(y).float(); self.p=torch.from_numpy(p).long()
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i], self.p[i]

loader = DataLoader(StateDataset(X_test, y_test, p_test), batch_size=64, shuffle=False)
all_logits, all_ys, all_ps = [], [], []
with torch.no_grad():
    for x, y, p in loader:
        all_logits.append(adapter(x.to(device)).cpu())
        all_ys.append(y); all_ps.append(p)
all_logits = torch.cat(all_logits).numpy()
all_ys = torch.cat(all_ys).numpy()
all_ps = torch.cat(all_ps).numpy()

adapter_aps = {}
for i, name in enumerate(PAIR_NAMES):
    mask = all_ps == i
    if mask.sum() < 2 or len(np.unique(all_ys[mask])) < 2: continue
    scores = torch.sigmoid(torch.from_numpy(all_logits[mask, i])).numpy()
    adapter_aps[name] = average_precision_score(all_ys[mask], scores)
    print(f'  {name:<16} adapter AP = {adapter_aps[name]:.3f}')

In [ ]:
# Results table — 4 methods, pre-stated verdicts
rows = []
for i, name in enumerate(PAIR_NAMES):
    n_tr   = int((p_train == i).sum())
    scarce = n_tr < SMALL_N
    zs = zs_aps.get(name); pr = probe_aps.get(name)
    st = stlm_aps.get(name); ad = adapter_aps.get(name)
    nl_delta = (st - pr) if (st is not None and pr is not None) else None  # non-linearity
    mt_delta = (ad - st) if (ad is not None and st is not None) else None  # multi-task
    rows.append({'pair': name, 'n_train': n_tr, 'scarce': scarce,
                 'zs_ap': zs, 'probe_ap': pr, 'stlm_ap': st, 'adapter_ap': ad,
                 'nl_delta': nl_delta, 'mt_delta': mt_delta})

df = pd.DataFrame(rows)
df_valid = df[df.pair != 'broken_intact']

print(f'{"Pair":<16} {"n_tr":>4} {"ZS AP":>6} {"LR":>6} {"ST-MLP":>7} {"MT-MLP":>7} {"NL Δ":>6} {"MT Δ":>6}')
print('-' * 70)
for r in df.itertuples():
    zs = f'{r.zs_ap:.3f}'   if r.zs_ap   is not None else '  n/a'
    pr = f'{r.probe_ap:.3f}' if r.probe_ap is not None else '  n/a'
    st = f'{r.stlm_ap:.3f}' if r.stlm_ap  is not None else '  n/a'
    ad = f'{r.adapter_ap:.3f}' if r.adapter_ap is not None else '  n/a'
    nl = f'{r.nl_delta:+.3f}' if r.nl_delta is not None else '   n/a'
    mt = f'{r.mt_delta:+.3f}' if r.mt_delta is not None else '   n/a'
    flag = ' SCARCE' if r.scarce else ''
    print(f'{r.pair:<16} {r.n_train:>4} {zs:>6} {pr:>6} {st:>7} {ad:>7} {nl:>6} {mt:>6}{flag}')

print('-' * 70)
print(f'{"Mean (excl broken)":<21}'
      f'{"":>6} {df_valid.probe_ap.mean():>6.3f} {df_valid.stlm_ap.mean():>7.3f}'
      f' {df_valid.adapter_ap.mean():>7.3f}')

print()
scarce_df = df_valid[df_valid.scarce]
if scarce_df.nl_delta.notna().any():
    nl_m = scarce_df.nl_delta.mean()
    mt_m = scarce_df.mt_delta.mean()
    print(f'Scarce-pair mean NL delta (LR→ST-MLP):    {nl_m:+.3f}')
    print(f'Scarce-pair mean MT delta (ST-MLP→MT-MLP): {mt_m:+.3f}')
    nl_v = 'NL matters' if nl_m > 0.05 else ('NL irrelevant' if nl_m < -0.05 else 'NL unclear')
    mt_v = 'MT CONFIRMED' if mt_m > 0.05 else ('MT FALSIFIED' if mt_m < -0.05 else 'MT INCONCLUSIVE')
    print(f'Non-linearity: {nl_v}')
    print(f'Multi-task:    {mt_v}')

In [ ]:
pairs_plot = [r.pair for r in df.itertuples() if r.pair != 'broken_intact' and r.probe_ap]
x = np.arange(len(pairs_plot)); w = 0.2

fig, ax = plt.subplots(figsize=(11, 4.5))
zs_v = [df[df.pair==p].zs_ap.values[0]      or 0 for p in pairs_plot]
pr_v = [df[df.pair==p].probe_ap.values[0]   or 0 for p in pairs_plot]
st_v = [df[df.pair==p].stlm_ap.values[0]    or 0 for p in pairs_plot]
ad_v = [df[df.pair==p].adapter_ap.values[0] or 0 for p in pairs_plot]

ax.bar(x - 1.5*w, zs_v, w, label='zero-shot',            color='#9B9B9B', alpha=0.85)
ax.bar(x - 0.5*w, pr_v, w, label='LR probe (linear)',     color='#4C72B0', alpha=0.85)
ax.bar(x + 0.5*w, st_v, w, label='ST-MLP (non-linear)',   color='#DD8452', alpha=0.85)
ax.bar(x + 1.5*w, ad_v, w, label='MT adapter (multi-task)',color='#C44E52', alpha=0.85)

for i, pair in enumerate(pairs_plot):
    if df[df.pair==pair].scarce.values[0]:
        ax.annotate('scarce', xy=(x[i], 0.02), ha='center', fontsize=7, color='orange')

ax.axhline(0.5, ls='--', color='grey', alpha=0.5, label='chance')
ax.set_xticks(x); ax.set_xticklabels(pairs_plot, rotation=15, fontsize=9)
ax.set_ylabel('Average Precision (AP)'); ax.set_ylim(0, 1.18)
ax.set_title('LR → ST-MLP isolates non-linearity | ST-MLP → MT-MLP isolates multi-task transfer\n'
             '(controlled: same embeddings, same split, same StandardScaler)')
ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

### Interpretation

| Measure | Pre-stated threshold | Observed | Verdict |
|---|---|---|---|
| Adapter vs probe on scarce pairs | > +5 AP delta | ___ | ___ |
| Adapter vs probe on rich pairs | within ±5 AP | ___ | ___ |
| Adapter vs zero-shot | Adapter > ZS on all pairs | ___ | ___ |

**Known limitation:** `broken_intact` excluded from mean AP. Only 10 positive training samples (target: 80). The adapter's broken_intact head is trained on severely imbalanced data and cannot be interpreted. Fix requires a floor-search strategy after `BreakObject` in `generate_sim_crops_all.py`.

**Comparison validity:** this is the most controlled comparison possible with current data. Both methods use the same frozen embeddings and the same split. Any remaining AP difference is attributable to the method, not to data selection.